Cargar train.csv → bronze_train
En el notebook 01_bronze_ingestion, ejecuta esta celda SQL:

In [0]:
%sql
INSERT INTO adbsflordeoro.default.bronze_train
SELECT
    CAST(id AS STRING),
    CAST(date AS STRING),
    CAST(store_nbr AS STRING),
    CAST(family AS STRING),
    CAST(sales AS STRING),
    CAST(onpromotion AS STRING)
FROM read_files(
    'abfss://raw@adlsgr18smartdata.dfs.core.windows.net/train.csv',
    format => 'csv',
    header => true,
    inferSchema => false
);

Esta tabla conservará los seis campos originales como STRING, tal como está definida actualmente.
2. Cargar transactions.csv → bronze_transactions_ext


In [0]:
%sql
INSERT INTO adbsflordeoro.default.bronze_transactions_ext
SELECT
    CAST(date AS DATE),
    CAST(store_nbr AS INT),
    CAST(transactions AS INT),
    'transactions.csv' AS _source_file,
    CURRENT_TIMESTAMP() AS _ingestion_timestamp,
    CURRENT_DATE() AS _ingestion_date
FROM read_files(
    'abfss://raw@adlsgr18smartdata.dfs.core.windows.net/transactions.csv',
    format => 'csv',
    header => true,
    inferSchema => false
);

Aquí convertimos los campos a sus tipos correspondientes y agregamos tres columnas de auditoría.
Cargar stores.csv → bronze_stores_ext
Ejecuta una tercera celda:


In [0]:
%sql
INSERT INTO adbsflordeoro.default.bronze_stores_ext
SELECT
    CAST(store_nbr AS INT),
    CAST(city AS STRING),
    CAST(state AS STRING),
    CAST(type AS STRING),
    CAST(cluster AS INT),
    'stores.csv' AS _source_file,
    CURRENT_TIMESTAMP() AS _ingestion_timestamp,
    CURRENT_DATE() AS _ingestion_date
FROM read_files(
    'abfss://raw@adlsgr18smartdata.dfs.core.windows.net/stores.csv',
    format => 'csv',
    header => true,
    inferSchema => false
);

validacion de la carga

In [0]:
%sql
SELECT 'bronze_train' AS tabla, COUNT(*) AS registros
FROM adbsflordeoro.default.bronze_train

UNION ALL

SELECT 'bronze_transactions_ext', COUNT(*)
FROM adbsflordeoro.default.bronze_transactions_ext

UNION ALL

SELECT 'bronze_stores_ext', COUNT(*)
FROM adbsflordeoro.default.bronze_stores_ext;

In [0]:
tables = [
    "adbsflordeoro.default.bronze_train",
    "adbsflordeoro.default.bronze_transactions_ext",
    "adbsflordeoro.default.bronze_stores_ext"
]

for table in tables:
    print(f"\n--- {table} ---")
    spark.sql(f"DESCRIBE DETAIL {table}") \
        .select("name", "format", "location", "numFiles", "sizeInBytes") \
        .show(truncate=False)